# Forward and backward stepwise modeling



## Setup


In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import KFold
from palmerpenguins import load_penguins
from plotnine import *
import statsmodels.api as sm

## Data


In [ ]:
penguins = load_penguins().dropna()
penguins.columns

## Regression

### Data Prep

In [ ]:
mass_features = ["bill_length_mm", "bill_depth_mm", "species", 
                 "flipper_length_mm", "island", "sex"]
X_mass = (
    pd.get_dummies(penguins[mass_features], dtype = int, drop_first=True)
) 
y_mass = penguins["body_mass_g"]

### Cross-Validation Setup

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

### Forward Selection Set Up

In [ ]:
selector = SequentialFeatureSelector(
    estimator=LinearRegression(),
    direction="forward",               
    n_features_to_select=4,
    scoring="neg_mean_squared_error",
    cv=cv,
)

### Selecting Predictors

In [ ]:
selector.fit(X_mass, y_mass)

### Selected Predictors 

In [ ]:
selected_features = X_mass.columns[selector.get_support()]
print(selected_features.tolist())

### Fit Model

In [ ]:
xx = sm.add_constant(X_mass[selected_features])
model = sm.OLS(y_mass, xx).fit()
model.summary()

### Problem

Fit a backward stepwise Regression model up 5 features. Use a 10-fold cross-validation approach.

## Classification

### Data Prep

In [ ]:
species_features = ["bill_length_mm", "bill_depth_mm", "body_mass_g", 
                 "flipper_length_mm"]
X_species = penguins[species_features]
y_species = pd.Categorical(penguins["species"])

### Cross Validation Set Up

In [ ]:
cv = KFold(n_splits=10, shuffle=True, random_state=42)

### Backward Selection Set Up

In [ ]:
selector = SequentialFeatureSelector(
    estimator=LogisticRegression(max_iter=10000),
    direction="backward",               
    n_features_to_select=2,
    scoring="accuracy",
    cv=cv,
)

### Selecting Predictors

In [ ]:
selector.fit(X_species, y_species)

### Selected Predictors

In [ ]:
selected_features = X_species.columns[selector.get_support()]
print(selected_features.tolist())

### Fitted Model

In [ ]:
xx = sm.add_constant(X_species[selected_features])
model = sm.MNLogit(y_species, xx).fit()
model.summary()